In [ ]:
# Cell 1 — Install dependencies
import subprocess
subprocess.run(["pip", "install", "flask", "flask-cors", "flask-sqlalchemy"])

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from flask_sqlalchemy import SQLAlchemy
from datetime import datetime

app = Flask(__name__)
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///ecocycle.db"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False
CORS(app)
db = SQLAlchemy(app)

class Product(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    name = db.Column(db.String(100), nullable=False)
    category = db.Column(db.String(50), nullable=False)
    expiry_date = db.Column(db.String(20))
    recycling_method = db.Column(db.String(100))
    weight_kg = db.Column(db.Float, default=0.5)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

    def to_dict(self):
        return {
            "id": self.id,
            "name": self.name,
            "category": self.category,
            "expiry_date": self.expiry_date,
            "recycling_method": self.recycling_method,
            "weight_kg": self.weight_kg
        }

RECYCLING_MAP = {
    "food":        {"method": "Composting",                     "co2_factor": 0.5},
    "dairy":       {"method": "Biogas production",              "co2_factor": 0.7},
    "oil":         {"method": "Oil recycling center",           "co2_factor": 1.1},
    "medicine":    {"method": "Pharmaceutical waste disposal",  "co2_factor": 0.3},
    "cosmetics":   {"method": "Chemical recycling",             "co2_factor": 0.4},
    "plastic":     {"method": "Plastic recycling bin",          "co2_factor": 0.6},
    "paper":       {"method": "Paper recycling",                "co2_factor": 0.2},
    "glass":       {"method": "Glass recycling bin",            "co2_factor": 0.3},
    "metal":       {"method": "Metal scrap recycling",          "co2_factor": 0.8},
    "clothing":    {"method": "Textile recycling center",       "co2_factor": 0.9},
    "electronics": {"method": "E-waste recycling center",       "co2_factor": 0.8},
    "battery":     {"method": "Battery drop-off point",         "co2_factor": 1.2},
    "chemicals":   {"method": "Hazardous waste disposal",       "co2_factor": 1.3},
    "paint":       {"method": "Paint recycling facility",       "co2_factor": 1.0},
}

CENTERS = [
    {"name": "EcoWaste Center",       "distance": "3km", "type": "General"},
    {"name": "GreenRecycle Hub",      "distance": "5km", "type": "Chemical"},
    {"name": "City Compost Plant",    "distance": "2km", "type": "Organic"},
    {"name": "PharmaDrop Point",      "distance": "4km", "type": "Medicine"},
    {"name": "E-Waste Collection",    "distance": "6km", "type": "Electronics"},
    {"name": "Textile Recycle Store", "distance": "3km", "type": "Clothing"},
]

def get_recycling_info(category):
    return RECYCLING_MAP.get(category.lower(), {"method": "General waste disposal", "co2_factor": 0.2})

@app.route("/api/products", methods=["POST"])
def add_product():
    data = request.json
    info = get_recycling_info(data["category"])
    product = Product(
        name=data["name"],
        category=data["category"],
        expiry_date=data.get("expiry_date"),
        recycling_method=info["method"],
        weight_kg=float(data.get("weight_kg", 0.5))
    )
    db.session.add(product)
    db.session.commit()
    return jsonify({"product": product.to_dict(), "recycling": info}), 201

@app.route("/api/products", methods=["GET"])
def get_products():
    products = Product.query.order_by(Product.created_at.desc()).all()
    return jsonify([p.to_dict() for p in products])

@app.route("/api/products/<int:id>", methods=["DELETE"])
def delete_product(id):
    product = Product.query.get(id)
    if not product:
        return jsonify({"error": "Not found"}), 404
    db.session.delete(product)
    db.session.commit()
    return jsonify({"message": "Deleted successfully"})

@app.route("/api/recycle/<category>", methods=["GET"])
def recycle_info(category):
    info = get_recycling_info(category)
    return jsonify({"method": info["method"], "centers": CENTERS})

@app.route("/api/stats", methods=["GET"])
def stats():
    products = Product.query.all()
    total_weight = sum(p.weight_kg for p in products)
    co2_saved = total_weight * 0.6
    counts = {}
    for p in products:
        counts[p.category] = counts.get(p.category, 0) + 1
    return jsonify({
        "total_products": len(products),
        "total_weight_kg": round(total_weight, 2),
        "co2_saved_kg": round(co2_saved, 2),
        "by_category": counts
    })

with app.app_context():
    db.create_all()

app.run(port=5000, debug=False, use_reloader=False)



 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [16/Mar/2026 11:26:16] "GET /api/stats HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:26:16] "GET /api/products HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:26:17] "GET /api/stats HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:26:17] "GET /api/products HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:26:18] "OPTIONS /api/products/1 HTTP/1.1" 200 -
C:\Users\tanisha jain\AppData\Local\Temp\ipykernel_23548\3266494496.py:82: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  product = Product.query.get(id)
127.0.0.1 - - [16/Mar/2026 11:26:19] "DELETE /api/products/1 HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:26:19] "GET /api/stats HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:26:19] "GET /api/products HTTP/